# Phase 12 — RDX-Zeuge: der Zeuge, den das Modell nicht abkoppeln kann

Ein Text-Zeuge kann wegoptimiert werden, weil das Modell ihn schreibt. Ein
Zeuge, den wir aus der **Repräsentationsdifferenz** ablesen, kann es nicht:
ihn zu eliminieren hieße, die Differenz aufzulösen — und das ist genau die
Verhaltensänderung.

Portiert aus [RDX](https://arxiv.org/abs/2505.23917) (NeurIPS 2025), Kern
gegen den Originalcode gegengerechnet. Layer 3 vs. Layer 23 am letzten
Prompt-Token, 120 Prompts, unüberwachte Cluster gegen die gemessene Kipprate,
k-means je Einzelschicht als Kontrolle.

Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~20 min.

In [ ]:
# === RDX-ZEUGE: ein Zeuge, den das Modell nicht abkoppeln kann ============
# Der Einwand gegen den Text-Zeugen (Nutzer): wenn eine verschriftlichte Spur
# das Modell wirklich bindet, haette jeder Optimierer sie eher ELIMINIERT als
# das Verhalten anzupassen - Protokoll und Ergebnis zu entkoppeln ist billiger.
# Ein wirksamer Zeuge braucht also Kosten(eliminieren) >> Kosten(anpassen).
#
# Genau diese Asymmetrie gilt fuer einen Zeugen, den nicht das Modell schreibt,
# sondern den WIR aus der Repraesentation ablesen. Es gibt nichts abzukoppeln:
# damit dieser Zeuge verschwindet, muss die Repraesentationsdifferenz selbst
# verschwinden - und das IST die Verhaltensaenderung. Deshalb ueberlebt er.
#
# Umgesetzt mit RDX (Representational Difference Explanations, NeurIPS 2025,
# Kondapaneni/Mac Aodha/Perona, arXiv 2505.23917). RDX vergleicht zwei
# Repraesentationen ueber denselben Datenpunkten und clustert das, was die eine
# anders gruppiert als die andere. Hier: Residuum am letzten Prompt-Token,
# Layer 3 (Anfang des Transportfensters) gegen Layer 23 (Ende) - Richtung "10"
# sind die Punkte, die die spaete Schicht NEU zusammengelegt hat.
#
# Der Zeuge steht, wenn diese unbeaufsichtigte Partition die gemessene Kipprate
# erklaert - RDX sieht die Labels nie. Kontrolle: k-means auf jeder Schicht
# einzeln. Schlaegt eine Einzelschicht die Differenz, braucht es RDX nicht.
# Statistik: eta^2 mit Permutationstest, ueber drei Clustersaaten, berichtet
# wird der Median und das SCHLECHTESTE p.
#
# Der RDX-Kern ist aus Erikiss/RDX @ a301a499 (src/rdx.py) nach numpy portiert:
# construct_graph(sim_function="neighborhood") + apply_diff_function
# ("locally_biased") + cluster_graph("spectral") + post_process, Parameter aus
# der v2-Config (beta=5, gamma=25/n, n_clusters=5, filter_thresh=1.1).
# Die Zelle klont das Repo und rechnet dieselbe Sache mit dem ORIGINALCODE
# nach - stimmt die Portierung nicht, faellt es sofort auf.
#
# Vorregistrierung Nr. 26: NICHT-DIFFERENZIELL ~35%, KEIN-ZEUGE ~35%,
# RDX-ZEUGE ~30%. Cell 19 fand keinen portablen Vektor; dass die Disposition
# am letzten Prompt-Token clusterlesbar ist, ist keineswegs gesetzt.
# Selbstversorgend: laeuft in einer FRISCHEN Runtime als einzige Zelle. ~20 min.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)

import matplotlib.pyplot as plt
N_PROMPTS=120; K_SAMP=6; MAX_NEW=20; MAXCHARS=1200; SEED=0
L_EARLY=3; L_LATE=23              # erster/letzter Voll-Attention-Layer im Transportfenster
BETA=5.0; GAMMA_SCALE=25.0        # v2-Config pinnt gamma=0.05 bei n=500 -> 25/n
N_CLUSTERS=5; FILTER_THRESH=1.1; ADD_NULL=True; N_PERM=2000
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------- RDX-Kern, portiert aus Erikiss/RDX @ a301a499 src/rdx.py --------
# construct_graph(sim_function="neighborhood") + apply_diff_function
# ("locally_biased") + cluster_graph("spectral") + post_process.
# numpy statt torch, damit die Logik ohne GPU testbar ist; identische Formeln.
def rdx_rank_dm(X):
    """cdist -> doppeltes argsort = Rangmatrix (skalenfrei), upstream Zeile 178-182"""
    X=np.asarray(X,dtype=np.float64)
    sq=(X*X).sum(1)
    D=np.sqrt(np.maximum(sq[:,None]+sq[None,:]-2*X@X.T,0.0))
    np.fill_diagonal(D,0.0)
    return np.argsort(np.argsort(D,axis=1),axis=1).astype(np.float64)
def rdx_diff(r1_dm,r0_dm,gamma,symmetrize="mean"):
    """upstream apply_diff_function, Zweig 'locally_biased'"""
    if symmetrize=="mean":
        r0_dm=(r0_dm+r0_dm.T)/2.0; r1_dm=(r1_dm+r1_dm.T)/2.0
    elif symmetrize=="max":
        r0_dm=np.maximum(r0_dm,r0_dm.T); r1_dm=np.maximum(r1_dm,r1_dm.T)
    denom=np.minimum(r1_dm,r0_dm)+1.0
    d10=np.tanh(gamma*(r1_dm-r0_dm)/denom)
    d01=np.tanh(gamma*(r0_dm-r1_dm)/denom)
    return d10,d01
def rdx_graph(X0,X1,beta=5.0,gamma=None,gamma_scale=25.0,symmetrize="mean"):
    """gibt die vier Matrizen zurueck, die upstream construct_graph liefert.
       Richtung '10': hohe Affinitaet = in X1 nah, in X0 fern -> vom spaeten
       Layer NEU gruppiert. Richtung '01': umgekehrt (auseinandergezogen)."""
    n=X0.shape[0]
    if gamma is None: gamma=gamma_scale/float(n)
    r0=rdx_rank_dm(X0); r1=rdx_rank_dm(X1)
    r0_am=np.exp(-beta*r0); r1_am=np.exp(-beta*r1)
    d10,d01=rdx_diff(r1,r0,gamma,symmetrize)
    return dict(am_10=np.exp(-beta*d10),am_01=np.exp(-beta*d01),
                r0_am=r0_am,r1_am=r1_am,diff_10=d10,diff_01=d01,gamma=gamma)
def rdx_post_process(labels,am,thresh=-1.0):
    """upstream post_process: Cluster nach mittlerer Innen-Affinitaet sortieren,
       alles unter thresh in den Null-Cluster 0."""
    labels=np.asarray(labels); means=[]
    uniq=np.unique(labels)
    for li in uniq:
        m=labels==li
        means.append(float(am[np.ix_(m,m)].mean()))
    means=np.array(means); order=np.argsort(means)
    new=np.zeros_like(labels)
    for i,ci in enumerate(order):
        new[labels==uniq[ci]]=0 if means[ci]<thresh else i+1
    return new,dict(zip(uniq.tolist(),means.tolist()))
def rdx_cluster(am,n_clusters,thresh=-1.0,seed=0,add_null_cluster=True):
    """upstream cluster_graph('spectral') inkl. Symmetrisierung und post_process"""
    from sklearn.cluster import SpectralClustering
    A=am.copy()
    if not np.allclose(A,A.T): A=(A+A.T)/2.0
    k=n_clusters+int(add_null_cluster)
    cl=SpectralClustering(n_clusters=k,affinity="precomputed",random_state=seed,
                          eigen_solver="arpack",assign_labels="kmeans",n_init=10)
    lab=cl.fit_predict(np.maximum(A,0.0))
    return rdx_post_process(lab,am,thresh)
def kmeans_labels(X,k,seed=0):
    """Kontrolle 'nur eine Schicht': k-means direkt auf der Repraesentation.
       Bewusst NICHT spektral auf exp(-beta*Rang) - diese Matrix ist praktisch
       diagonal (exp(-5)=0.007) und die Clusterung darauf springt zwischen
       Laeufen; k-means ist stabil und als Baseline fair."""
    from sklearn.cluster import KMeans
    Xs=(X-X.mean(0))/ (X.std(0)+1e-9)
    return KMeans(n_clusters=k,random_state=seed,n_init=10).fit_predict(Xs)+1
# ---------- Auswertung: erklaert die Partition das Kippverhalten? -----------
def eta_squared(labels,y):
    """Anteil der Varianz der Kipprate, den die Partition erklaert"""
    y=np.asarray(y,dtype=np.float64); labels=np.asarray(labels)
    tot=((y-y.mean())**2).sum()
    if tot<=0: return 0.0
    bet=0.0
    for li in np.unique(labels):
        m=labels==li
        bet+=m.sum()*(y[m].mean()-y.mean())**2
    return float(bet/tot)
def perm_p(labels,y,n_perm=2000,seed=0):
    """Permutationstest auf eta^2 - kalibriert gegen die Clusterzahl"""
    rng=np.random.default_rng(seed)
    obs=eta_squared(labels,y); y=np.asarray(y,dtype=np.float64)
    ge=0
    for _ in range(n_perm):
        if eta_squared(labels,rng.permutation(y))>=obs: ge+=1
    return obs,(1.0+ge)/(1.0+n_perm)
def cluster_table(labels,y,extra=None):
    """je Cluster: Groesse, mittlere Kipprate, optionale Zusatzspalten"""
    out=[]
    for li in np.unique(labels):
        m=np.asarray(labels)==li
        row=dict(label=int(li),n=int(m.sum()),rate=float(np.mean(np.asarray(y)[m])))
        if extra:
            for k,v in extra.items(): row[k]=float(np.mean(np.asarray(v)[m]))
        out.append(row)
    return sorted(out,key=lambda r:-r["rate"])
def stability(pairs):
    """aus (eta,p) je Saat: konservativ zusammengefasst - Median-eta, schlechtestes p"""
    es=sorted(x[0] for x in pairs); ps=[x[1] for x in pairs]
    med=es[len(es)//2] if len(es)%2 else (es[len(es)//2-1]+es[len(es)//2])/2
    return dict(eta=med,eta_lo=es[0],eta_hi=es[-1],p=max(ps),spread=es[-1]-es[0])
def verdict_rdx(p_rdx,eta_rdx,eta_r0,eta_r1,alpha=0.05,margin=1.15):
    """RDX-ZEUGE nur, wenn die Differenzstruktur signifikant ist UND beide
       Einzelschichten schlaegt - sonst steckt die Struktur schon in einer
       Repraesentation und die Differenz erklaert nichts Eigenes."""
    if p_rdx>=alpha: return "KEIN-ZEUGE"
    if eta_rdx>margin*max(eta_r0,eta_r1): return "RDX-ZEUGE"
    return "NICHT-DIFFERENZIELL"
# ---------------- Klassifikator (identisch zu Phase 11/12) ------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
SW=("takeover","gloss","latin-switch(fr)")
KOED=["local name","native name","original name","local language","own language",
      "romaji","kanji","hiragana","katakana","transliterat","native script",
      "local script","in japanese","in chinese","in korean","original language",
      "mother tongue","local term","native term"]
def has_koeder(p):
    pl=p.lower(); return float(any(k in pl for k in KOED))
# ---------------- Quervergleich gegen den Upstream-Code ---------------------
def upstream_crosscheck(X0,X1,beta,gamma,ours):
    """laedt Erikiss/RDX, schneidet construct_graph heraus und rechnet dieselbe
       Sache mit dem Originalcode - Portierungsfehler faellt sofort auf."""
    import subprocess, textwrap
    d="/content/rdx_upstream"
    if not os.path.isdir(os.path.join(d,"src")):
        r=subprocess.run(["git","clone","--depth","1","https://github.com/Erikiss/RDX",d],
                         capture_output=True,text=True,timeout=600)
        if r.returncode!=0:
            return None,"Klon fehlgeschlagen (privates Repo?)"
    p=os.path.join(d,"src","rdx.py")
    if not os.path.exists(p): return None,"src/rdx.py nicht gefunden"
    src=open(p,encoding="utf-8").read()
    def grab(name):
        i=src.find("    def %s("%name)
        if i<0: return None
        ends=[x for x in (src.find("\n    @staticmethod",i),src.find("\n    def ",i+10)) if x>0]
        return textwrap.dedent(src[i:(min(ends) if ends else len(src))])
    parts=[grab(n) for n in ("apply_guid_labels","apply_diff_function","construct_graph")]
    if any(x is None for x in parts): return None,"Funktionen nicht extrahierbar"
    g={"torch":torch,"np":np}
    exec("\n".join(parts),g)
    class _R: pass
    _R.apply_guid_labels=staticmethod(g["apply_guid_labels"])
    _R.apply_diff_function=staticmethod(g["apply_diff_function"])
    g["RDX"]=_R
    out=g["construct_graph"](X0,X1,dict(sim_function="neighborhood",guidance=None,
        diff_function="locally_biased",symmetrize_dm="mean",beta=beta,gamma=gamma,
        normalize_diff_mat_by_abs_max=False))
    dev=max(float(np.abs(out[k].numpy()-ours[k]).max()) for k in ("am_10","am_01","diff_10","diff_01"))
    return dev,"ok"
# ---------------- Daten: Repraesentationen + Kipprate je Prompt -------------
rng=np.random.default_rng(SEED)
cand=[p for p in PROMPT_IDS if 0<len(PROMPTS[p])<=MAXCHARS]
sel=[cand[i] for i in rng.permutation(len(cand))[:N_PROMPTS]]
print("RDX-Zeuge | %d Prompts (von %d brauchbaren), Layer %d vs. %d, K=%d Ziehungen"
      %(len(sel),len(cand),L_EARLY,L_LATE,K_SAMP))
NL=model.config.num_hidden_layers
assert L_LATE<NL, "L_LATE ausserhalb des Modells"
@torch.no_grad()
def repr_pair(u):
    ids=tokenizer(think_prefix(u),return_tensors="pt").input_ids.to(model.device)
    hs=model(input_ids=ids,output_hidden_states=True).hidden_states
    return (hs[L_EARLY+1][0,-1].float().cpu().numpy(),
            hs[L_LATE +1][0,-1].float().cpu().numpy())
@torch.no_grad()
def switch_rate(u):
    ids=tokenizer(think_prefix(u),return_tensors="pt").input_ids.to(model.device)
    o=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                     repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                     num_return_sequences=K_SAMP,pad_token_id=tokenizer.eos_token_id)
    cls=[classify_answer(tokenizer.decode(x[ids.shape[1]:],skip_special_tokens=True)) for x in o]
    return sum(1 for c in cls if c in SW)/float(K_SAMP)
E=[];L=[];Y=[]
for i,pid in enumerate(sel):
    u=PROMPTS[pid]
    a,b=repr_pair(u); E.append(a); L.append(b); Y.append(switch_rate(u))
    if (i+1)%20==0: print("  %3d/%d | bisherige mittlere Kipprate %.3f"%(i+1,len(sel),np.mean(Y)))
E=np.stack(E); L=np.stack(L); Y=np.array(Y)
nz=int((Y>0).sum())
print("  Kipprate: Mittel %.3f | Prompts mit mindestens einem Kipp %d/%d | Spanne %.2f..%.2f"
      %(Y.mean(),nz,len(Y),Y.min(),Y.max()))
if nz<15:
    print("  !! WARNUNG: nur %d Prompts kippen ueberhaupt - der Test ist unterversorgt,"%nz)
    print("     jedes Nullergebnis unten ist ein Dosis- und kein Sachbefund.")
# ---------------- RDX ------------------------------------------------------
G=rdx_graph(E,L,beta=BETA,gamma_scale=GAMMA_SCALE)
print("\nRDX-Graph gebaut (beta=%.1f, gamma=%.4f, symmetrisierte Rangabstaende)"%(BETA,G["gamma"]))
dev,msg=(None,"uebersprungen")
try: dev,msg=upstream_crosscheck(E,L,BETA,G["gamma"],G)
except Exception as e: msg="Fehler: %s: %s"%(type(e).__name__,str(e)[:60])
print("  Quervergleich mit dem Originalcode: %s"
      %("maximale Abweichung %.2e"%dev if dev is not None else msg))
KTOT=N_CLUSTERS+int(ADD_NULL)
MAKE={"RDX 10 (spaet neu gruppiert)":lambda s: rdx_cluster(G["am_10"],N_CLUSTERS,FILTER_THRESH,s,ADD_NULL)[0],
      "RDX 01 (spaet getrennt)":     lambda s: rdx_cluster(G["am_01"],N_CLUSTERS,FILTER_THRESH,s,ADD_NULL)[0],
      "nur Layer %d"%L_EARLY:        lambda s: kmeans_labels(E,KTOT,s),
      "nur Layer %d"%L_LATE:         lambda s: kmeans_labels(L,KTOT,s)}
NMS=list(MAKE)
SEEDS=[SEED,SEED+1,SEED+2]
print("\nErklaerte Varianz der Kipprate (eta^2), Permutationstest mit %d Ziehungen,"%N_PERM)
print("ueber %d Clustersaaten - berichtet werden Median-eta^2 und das SCHLECHTESTE p:"%len(SEEDS))
STAT={}; LABS={}
for name in NMS:
    pairs=[]
    for s in SEEDS:
        lb=MAKE[name](s); pairs.append(perm_p(lb,Y,N_PERM,SEED))
        if s==SEED: LABS[name]=lb
    st=stability(pairs); STAT[name]=st
    print("  %-28s eta^2=%.4f [%.4f..%.4f]  p=%.4f  (%d Cluster)"
          %(name,st["eta"],st["eta_lo"],st["eta_hi"],st["p"],len(np.unique(LABS[name]))))
lab10=LABS["RDX 10 (spaet neu gruppiert)"]; lab01=LABS["RDX 01 (spaet getrennt)"]
eta10=STAT["RDX 10 (spaet neu gruppiert)"]["eta"]; p10=STAT["RDX 10 (spaet neu gruppiert)"]["p"]
eta_r0=STAT["nur Layer %d"%L_EARLY]["eta"]; eta_r1=STAT["nur Layer %d"%L_LATE]["eta"]
if STAT["RDX 10 (spaet neu gruppiert)"]["spread"]>0.5*max(eta10,1e-9):
    print("  !! die RDX-Partition ist saatabhaengig (Spanne %.3f) - das Verdikt unten"
          %STAT["RDX 10 (spaet neu gruppiert)"]["spread"])
    print("     steht auf wackligem Grund, mehr Saaten oder mehr Prompts noetig.")
# ---------------- Was sagt der Zeuge aus? ----------------------------------
LENS=np.array([len(PROMPTS[p]) for p in sel],dtype=float)
KOE =np.array([has_koeder(PROMPTS[p]) for p in sel])
SHORT=(LENS<40).astype(float)
tab=cluster_table(lab10,Y,extra={"laenge":LENS,"koeder":KOE,"kurz":SHORT})
print("\nRDX-Cluster der Richtung 10 (was Layer %d..%d neu zusammengelegt hat):"%(L_EARLY,L_LATE))
print("  %-7s %4s %8s %9s %8s %7s"%("Cluster","n","Kipprate","Zeichen","Koeder","kurz"))
for r in tab:
    print("  %-7d %4d %8.3f %9.0f %8.2f %7.2f"
          %(r["label"],r["n"],r["rate"],r["laenge"],r["koeder"],r["kurz"]))
top=tab[0]["label"]; bot=tab[-1]["label"]
for nm,cl in (("staerkster Kipp-Cluster",top),("schwaechster",bot)):
    idx=[i for i in range(len(sel)) if lab10[i]==cl][:3]
    print("\n  %s (Cluster %d, Kipprate %.2f) - Beispiele:"
          %(nm,cl,float(np.mean(Y[lab10==cl]))))
    for i in idx:
        t=" ".join(PROMPTS[sel[i]].split())
        print("    [%.2f] %s"%(Y[i],t[:96]+("…" if len(t)>96 else "")))
# ---------------- Karten ----------------------------------------------------
try:
    from sklearn.manifold import SpectralEmbedding
    emb=SpectralEmbedding(n_components=2,affinity="precomputed",random_state=SEED
                          ).fit_transform(np.maximum((G["am_10"]+G["am_10"].T)/2,0))
except Exception:
    emb=None
fig=plt.figure(figsize=(15,4.6)); gs=fig.add_gridspec(1,3,wspace=.28)
ax=fig.add_subplot(gs[0,0])
if emb is not None:
    sc=ax.scatter(emb[:,0],emb[:,1],c=Y,cmap="viridis",s=42,edgecolor="#222",linewidth=.4)
    plt.colorbar(sc,ax=ax,fraction=.046,label="Kipprate")
    for cl in np.unique(lab10):
        m=lab10==cl
        ax.annotate("C%d"%cl,(emb[m,0].mean(),emb[m,1].mean()),fontsize=11,fontweight="bold",
                    color="#B91C1C",ha="center")
else:
    ax.text(.5,.5,"Einbettung nicht verfuegbar",ha="center")
ax.set_title("RDX-Differenzgraph (Richtung 10)\nFarbe = Kipprate, C = Cluster",fontsize=10)
ax.set_xlabel("Spektralkoordinate 1"); ax.set_ylabel("Spektralkoordinate 2")
ax2=fig.add_subplot(gs[0,1])
cls=[r["label"] for r in tab]; rts=[r["rate"] for r in tab]
ses=[float(np.std(Y[lab10==c],ddof=1)/max(np.sqrt((lab10==c).sum()),1)) if (lab10==c).sum()>1
     else 0.0 for c in cls]
ax2.bar(range(len(cls)),rts,yerr=[1.96*s for s in ses],capsize=4,
        color=["#DC2626" if r>Y.mean() else "#2563EB" for r in rts])
ax2.axhline(Y.mean(),ls="--",c="#666",lw=1,label="Gesamtmittel %.3f"%Y.mean())
ax2.set_xticks(range(len(cls))); ax2.set_xticklabels(["C%d\nn=%d"%(c,r["n"]) for c,r in zip(cls,tab)],fontsize=8)
ax2.set_ylabel("Kipprate"); ax2.set_title("Kipprate je RDX-Cluster (95 %)",fontsize=10)
ax2.legend(frameon=False,fontsize=8)
ax3=fig.add_subplot(gs[0,2])
nms=NMS; es=[STAT[n]["eta"] for n in nms]; pv=[STAT[n]["p"] for n in nms]
err=[[STAT[n]["eta"]-STAT[n]["eta_lo"] for n in nms],[STAT[n]["eta_hi"]-STAT[n]["eta"] for n in nms]]
ax3.barh(range(len(nms))[::-1],es,xerr=err,capsize=3,
         color=["#DC2626","#F59E0B","#9CA3AF","#9CA3AF"])
for i,(e,p) in enumerate(zip(es,pv)):
    ax3.text(STAT[nms[i]]["eta_hi"]+.006,len(nms)-1-i,"p=%.3f"%p,va="center",fontsize=8)
ax3.set_yticks(range(len(nms))[::-1]); ax3.set_yticklabels(nms,fontsize=8)
ax3.set_xlabel("eta² (erklaerte Varianz der Kipprate)")
ax3.set_title("Differenzstruktur gegen Einzelschichten\n(Balken: Median, Fehler: Saatspanne)",fontsize=9)
ax3.set_xlim(0,max([STAT[n]["eta_hi"] for n in nms])*1.4+.02)
plt.show()
# ---------------- Verdikt ---------------------------------------------------
code=verdict_rdx(p10,eta10,eta_r0,eta_r1)
print("\nVERDIKT:",end=" ")
if code=="RDX-ZEUGE":
    print("RDX-ZEUGE STEHT: die unbeaufsichtigte Differenzstruktur zwischen Layer %d"%L_EARLY)
    print("  und %d erklaert die Kipprate (eta^2=%.3f, p=%.4f) und schlaegt beide"%(L_LATE,eta10,p10))
    print("  Einzelschichten (%.3f / %.3f). Der Zeuge steht in der Repraesentations-"%(eta_r0,eta_r1))
    print("  DIFFERENZ, nicht im Text: das Modell schreibt ihn nicht, also kann es ihn")
    print("  auch nicht abkoppeln. Ihn zu eliminieren hiesse, die Differenz aufzuloesen -")
    print("  und das ist genau die Verhaltensaenderung. Kosten(eliminieren) =")
    print("  Kosten(anpassen); der Sonderfall aus Deinem Einwand ist hier der Regelfall.")
elif code=="NICHT-DIFFERENZIELL":
    print("ZEUGE JA, DIFFERENZ NEIN: die Partition erklaert die Kipprate (eta^2=%.3f,"%eta10)
    print("  p=%.4f), aber eine einzelne Schicht tut es genauso gut (%.3f / %.3f)."%(p10,eta_r0,eta_r1))
    print("  Ablesbar ist die Disposition also schon in EINER Repraesentation; der")
    print("  RDX-Schritt fuegt nichts hinzu. Der Zeuge ueberlebt, aber er braucht")
    print("  keine Differenzerklaerung - die einfachere Sonde genuegt.")
else:
    p_r0=STAT["nur Layer %d"%L_EARLY]["p"]; p_r1=STAT["nur Layer %d"%L_LATE]["p"]
    if min(p_r0,p_r1)<0.05:
        print("ZEUGE JA, ABER NICHT IN DER DIFFERENZ: der RDX-Graph erklaert nichts")
        print("  (eta^2=%.3f, p=%.4f), eine EINZELNE Schicht dagegen schon"%(eta10,p10))
        print("  (Layer %d: eta^2=%.3f p=%.4f | Layer %d: eta^2=%.3f p=%.4f)."
              %(L_EARLY,eta_r0,p_r0,L_LATE,eta_r1,p_r1))
        print("  Die Disposition ist unbeaufsichtigt ablesbar - der Zeuge steht also,")
        print("  und er ist nicht abkoppelbar. Nur braucht es dafuer keine Differenz-")
        print("  erklaerung: die Repraesentation selbst genuegt als Protokoll.")
    else:
        print("KEIN ZEUGE AN DIESER STELLE: keine Partition erklaert die Kipprate")
        print("  signifikant (RDX eta^2=%.3f, p=%.4f; Einzelschichten %.3f / %.3f)."%(eta10,p10,eta_r0,eta_r1))
        print("  Am letzten Prompt-Token ist die Disposition nicht clusterlesbar - was zu")
        print("  Cell 19 passt (kein portabler Vektor). Naechster Ansatz: Position des")
        print("  Koeders statt letztes Token, oder KV-Zustaende statt Residuum.")
print("(Gelesen wird das Residuum am LETZTEN Prompt-Token - dem Zustand, der das")
print(" erste Antwort-Token erzeugt. Das Tor liegt eine Position spaeter.)")
RDX_RESULTS=dict(verdict=code,stats=dict((n,STAT[n]) for n in nms),n=len(sel),
                 rate_mean=float(Y.mean()),clusters=tab,
                 upstream_dev=dev,layers=(L_EARLY,L_LATE))
